In [1]:
import numpy as np
import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam

import random

In [2]:
np.random.seed(42)

X = np.random.uniform(
    low=[150, 5, 10],
    high=[200, 15, 30],
    size=(100, 3)
)

# Moisture content (target)
y = (
    0.02 * X[:, 0]
    - 0.5 * X[:, 1]
    + 0.1 * X[:, 2]
    + np.random.randn(100)
).reshape(-1, 1)

# Normalize data
X = X / np.max(X, axis=0)

y = y / np.max(y)

In [3]:
def fitness_function(params):

    neurons = int(params[0])

    learning_rate = params[1]

    model = Sequential([
        Dense(
            neurons,
            activation='relu',
            input_shape=(3,)
        ),

        Dense(1)
    ])

    model.compile(
        optimizer=Adam(
            learning_rate=learning_rate
        ),

        loss='mse'
    )

    model.fit(
        X,
        y,
        epochs=50,
        verbose=0
    )

    loss = model.evaluate(
        X,
        y,
        verbose=0
    )

    return loss

In [4]:
population_size = 10

generations = 10

# Chromosome = [neurons, learning_rate]
population = [

    [
        random.randint(5, 50),

        random.uniform(0.001, 0.05)
    ]

    for _ in range(population_size)
]

In [5]:
# Selection
def selection(pop, scores):

    selected = pop[np.argmin(scores)]

    return selected


# Crossover
def crossover(parent1, parent2):

    return [

        random.choice([
            parent1[0],
            parent2[0]
        ]),

        random.choice([
            parent1[1],
            parent2[1]
        ])
    ]


# Mutation
def mutation(child):

    if random.random() < 0.2:
        child[0] = random.randint(5, 50)

    if random.random() < 0.2:
        child[1] = random.uniform(0.001, 0.05)

    return child

In [6]:
for gen in range(generations):

    scores = [
        fitness_function(ind)
        for ind in population
    ]

    best = selection(
        population,
        scores
    )

    new_population = []

    for _ in range(population_size):

        parent2 = random.choice(population)

        child = crossover(
            best,
            parent2
        )

        child = mutation(child)

        new_population.append(child)

    population = new_population

    print(
        f"Generation {gen + 1}, "
        f"Best Loss: {min(scores):.5f}"
    )

D:\ANACONDA\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Generation 1, Best Loss: 0.03961
Generation 2, Best Loss: 0.03974
Generation 3, Best Loss: 0.03936
Generation 4, Best Loss: 0.03972
Generation 5, Best Loss: 0.03877
Generation 6, Best Loss: 0.03926
Generation 7, Best Loss: 0.03961
Generation 8, Best Loss: 0.03871
Generation 9, Best Loss: 0.03938
Generation 10, Best Loss: 0.04026


In [7]:
best_params = selection(

    population,

    [
        fitness_function(ind)
        for ind in population
    ]
)

print(
    "Optimized Number of Neurons:",
    int(best_params[0])
)

print(
    "Optimized Learning Rate:",
    best_params[1]
)

Optimized Number of Neurons: 34
Optimized Learning Rate: 0.020571133173398794
